# PDF Document Loaders
- Load various kind of documents from the web and local files.
- Apply LLM to the documents for summarization and question answering.
## Project 1: Question Answering from PDF Document
- We will load the document from the local file and apply LLM to answer the questions.
- Lets use research paper published on the missuse of the health supplements for workout.

In [1]:
!git clone git@github.com:laxmimerit/rag-dataset.git
# !pip install pymupdf tiktoken 

Cloning into 'rag-dataset'...
remote: Enumerating objects: 1277, done.
remote: Counting objects: 100% (3/3), done.
remote: Compressing objects: 100% (2/2), done.
remote: Total 1277 (delta 2), reused 1 (delta 1), pack-reused 1274 (from 2)
Receiving objects: 100% (1277/1277), 47.06 MiB | 2.79 MiB/s, done.
Resolving deltas: 100% (607/607), done.


In [2]:
from dotenv import load_dotenv

load_dotenv()

False

In [5]:
!pip install pymupdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 3.9 MB/s  0:00:063.8 MB/s eta 0:00:01:01


In [7]:
from langchain_community.document_loaders import PyMuPDFLoader

loader = PyMuPDFLoader("rag-dataset/health supplements/1. dietary supplements - for whom.pdf")

docs = loader.load()

In [8]:
len(docs)

17

In [9]:
# docs[0].metadata
# print(docs[0].page_content)

In [11]:
### Read the list of PDFs in the dir
import os

pdfs = []
for root, dirs, files in os.walk('rag-dataset'):
    # print(root, dirs, files)
    for file in files:
        if file.endswith(".pdf"):
            pdfs.append(os.path.join(root, file))

In [13]:
docs = []

for pdf in pdfs:
    loader = PyMuPDFLoader(pdf)
    temp=loader.load()
    docs.extend(temp)

    # print(temp)
    # break

In [14]:
len(docs)

1760

In [16]:
def format_docs(docs):
    return "\n\n".join([x.page_content for x in docs])

context = format_docs(docs)

In [17]:
docs[0]

Document(metadata={'producer': 'EDGRpdf Service w/ EO.Pdf 22.0.40.0', 'creator': 'EDGAR Filing HTML Converter', 'creationdate': '2023-11-03T06:02:32-04:00', 'source': 'rag-dataset/finance/pdfs/apple/apple 10-k 2023.pdf', 'file_path': 'rag-dataset/finance/pdfs/apple/apple 10-k 2023.pdf', 'total_pages': 80, 'format': 'PDF 1.4', 'title': '0000320193-23-000106', 'author': 'EDGAR® Online LLC, a subsidiary of OTC Markets Group', 'subject': 'Form 10-K filed on 2023-11-03 for the period ending 2023-09-30', 'keywords': '0000320193-23-000106; ; 10-K', 'moddate': '2023-11-03T06:02:40-04:00', 'trapped': '', 'encryption': 'Standard V2 R3 128-bit RC4', 'modDate': "D:20231103060240-04'00'", 'creationDate': "D:20231103060232-04'00'", 'page': 0}, page_content='UNITED STATES\nSECURITIES AND EXCHANGE COMMISSION\nWashington, D.C. 20549\nFORM 10-K\n(Mark One)\n☒ ANNUAL REPORT PURSUANT TO SECTION 13 OR 15(d) OF THE SECURITIES EXCHANGE ACT OF 1934\nFor the fiscal year ended September\xa030, 2023\nor\n☐ TRANS

In [18]:
# print(context)

In [ ]:
import tiktoken

encoding = tiktoken.encoding_for_model("gpt-4o-mini")

In [ ]:
encoding.encode("congratulations"), encoding.encode("rqsqeft")

In [ ]:
len(encoding.encode(docs[0].page_content))

In [ ]:
len(encoding.encode(context))

In [ ]:
969*64

In [ ]:
### Question Answering using LLM
from langchain_ollama import ChatOllama

from langchain_core.prompts import (SystemMessagePromptTemplate, HumanMessagePromptTemplate,
                                    ChatPromptTemplate)



from langchain_core.output_parsers import StrOutputParser

base_url = "http://localhost:11434"
model = 'qwen3'

llm = ChatOllama(base_url=base_url, model=model)

In [ ]:
system = SystemMessagePromptTemplate.from_template("""You are helpful AI assistant who answer user question based on the provided context. 
                                                    Do not answer in more than {words} words""")

prompt = """Answer user question based on the provided context ONLY! If you do not know the answer, just say "I don't know".
            ### Context:
            {context}

            ### Question:
            {question}

            ### Answer:"""

prompt = HumanMessagePromptTemplate.from_template(prompt)

messages = [system, prompt]
template = ChatPromptTemplate(messages)

# template
# template.invoke({'context': context, 'question': "How to gain muscle mass?", 'words': 50})

qna_chain = template | llm | StrOutputParser()


In [ ]:
qna_chain

In [ ]:
response = qna_chain.invoke({'context': context, 'question': "How to gain muscle mass?", 'words': 50})
print(response)

In [ ]:
response = qna_chain.invoke({'context': context, 'question': "How to reduce the weight?", 'words': 50})
print(response)

In [ ]:
response = qna_chain.invoke({'context': context, 'question': "How to do weight loss?", 'words': 50})
print(response)

In [ ]:
response = qna_chain.invoke({'context': context, 'question': "How many planets are there outside of our solar system?", 'words': 50})
print(response)

## Project 2: PDF Document Summarization

In [ ]:
system = SystemMessagePromptTemplate.from_template("""You are helpful AI assistant who works as document summarizer. 
                                                   You must not hallucinate or provide any false information.""")

prompt = """Summarize the given context in {words}.
            ### Context:
            {context}

            ### Summary:"""

prompt = HumanMessagePromptTemplate.from_template(prompt)

messages = [system, prompt]
template = ChatPromptTemplate(messages)

summary_chain = template | llm | StrOutputParser()

In [ ]:
summary_chain

In [ ]:
response = summary_chain.invoke({'context': context, 'words': 50})
print(response)

In [ ]:
response = summary_chain.invoke({'context': context, 'words': 500})
print(response)

## Project 3: Report Generation from PDF Document

In [ ]:
response = qna_chain.invoke({'context': context, 
                             'question': "Provide a detailed report from the provided context. Write answer in Markdown.", 
                             'words': 2000})
print(response)